In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import os
import optuna
import shap

from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, cross_val_predict, train_test_split,
)
from sklearn.metrics import accuracy_score, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


In [2]:
path = './data/'
train_data = pd.read_csv(path + 'train.csv')
test_data  = pd.read_csv(path + 'test.csv')


In [3]:
features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges',
]


## Baseline Models

In [4]:
train_df = train_data.copy()
test_df  = test_data.copy()

# String features with exactly 2 distinct values — one dummy column suffices.
# drop_first=True drops the alphabetically-first category:
#   gender → gender_Male,  Partner/Dependents/PhoneService/PaperlessBilling → <col>_Yes
binary_str_cols = [f for f in features
                   if pd.api.types.is_string_dtype(train_data[f]) and train_data[f].nunique() <= 2]

# String features with 3+ distinct values — keep all dummies (no reference-category drop).
multi_str_cols = [f for f in features
                  if pd.api.types.is_string_dtype(train_data[f]) and train_data[f].nunique() >= 3]

# Binary encode
train_df = pd.get_dummies(train_df, columns=binary_str_cols, drop_first=True, dtype=int)
test_df  = pd.get_dummies(test_df,  columns=binary_str_cols, drop_first=True, dtype=int)

# Churn: Yes → 1, No → 0  (test_data has no Churn column, so no action needed there)
train_df['Churn'] = train_df['Churn'].map({'Yes': 1, 'No': 0})

# One-hot encode multi-class string features
train_df = pd.get_dummies(train_df, columns=multi_str_cols, dtype=int)
test_df  = pd.get_dummies(test_df,  columns=multi_str_cols, dtype=int)

print("train_df shape:", train_df.shape)
print("test_df  shape:", test_df.shape)

train_df shape: (594194, 42)
test_df  shape: (254655, 41)


In [5]:
# All columns in train_df that descend from `features` — i.e. everything except id and Churn.
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
print(f"{len(encoded_features)} features:")
encoded_features

40 features:


['SeniorCitizen',
 'tenure',
 'MonthlyCharges',
 'TotalCharges',
 'gender_Male',
 'Partner_Yes',
 'Dependents_Yes',
 'PhoneService_Yes',
 'PaperlessBilling_Yes',
 'MultipleLines_No',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'PaymentMethod_Bank transfer (automatic)',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',

In [6]:
# X = train_df[encoded_features]
# y = train_df['Churn']

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# max_depths = range(1, 16)

# depth_accuracies = [
#     cross_val_score(LGBMClassifier(max_depth=d, num_leaves=2**d, verbose=-1), X, y, cv=cv, scoring='accuracy').mean()
#     for d in max_depths
# ]

# plt.figure(figsize=(7, 4))
# plt.plot(list(max_depths), depth_accuracies, marker='o')
# plt.xlabel('max_depth')
# plt.ylabel('CV Accuracy (5-fold mean)')
# plt.title('LGBMClassifier: max_depth vs CV Accuracy')
# plt.xticks(list(max_depths))
# plt.tight_layout()
# plt.show()

In [7]:
# depth_accuracies_xgb = [
#     cross_val_score(XGBClassifier(max_depth=d, verbosity=0, eval_metric='logloss'), X, y, cv=cv, scoring='accuracy').mean()
#     for d in max_depths
# ]

# plt.figure(figsize=(7, 4))
# plt.plot(list(max_depths), depth_accuracies_xgb, marker='o', color='orange')
# plt.xlabel('max_depth')
# plt.ylabel('CV Accuracy (5-fold mean)')
# plt.title('XGBClassifier: max_depth vs CV Accuracy')
# plt.xticks(list(max_depths))
# plt.tight_layout()
# plt.show()

In [8]:

X = train_df[encoded_features]
y = train_df['Churn']

# Constant predictor: always predict the majority class
baseline_acc = max(y.mean(), 1 - y.mean())
print(f"Constant predictor accuracy: {baseline_acc:.4f}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic regression inside a Pipeline so StandardScaler is fit only on
# training folds — prevents any leakage of validation-fold statistics.
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42)),
])

lr_scores   = cross_val_score(lr_pipeline,       X, y, cv=cv, scoring='accuracy')
lgbm_scores = cross_val_score(LGBMClassifier(verbose=-1), X, y, cv=cv, scoring='accuracy')

print(f"\nLogistic Regression  — CV accuracy: {lr_scores.mean():.4f} ± {lr_scores.std():.4f}")
print(f"LGBMClassifier       — CV accuracy: {lgbm_scores.mean():.4f} ± {lgbm_scores.std():.4f}")

Constant predictor accuracy: 0.7748

Logistic Regression  — CV accuracy: 0.8545 ± 0.0010
LGBMClassifier       — CV accuracy: 0.8603 ± 0.0006


In [9]:
lr_auc_scores = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression  — CV ROC-AUC: {lr_auc_scores.mean():.4f} ± {lr_auc_scores.std():.4f}")

print("\nLogistic Regression default parameters:")
pd.Series(LogisticRegression().get_params()).to_frame('value')

Logistic Regression  — CV ROC-AUC: 0.9079 ± 0.0011

Logistic Regression default parameters:


,value
C,1.0
class_weight,None
dual,False
fit_intercept,True
intercept_scaling,1
l1_ratio,0.0
max_iter,100
n_jobs,None
penalty,deprecated
random_state,None


In [10]:
# OOF probabilities needed for best_threshold (used in Save Artifacts).
oof_proba = cross_val_predict(lr_pipeline, X, y, cv=cv, method='predict_proba')[:, 1]

# --- ROC curve (commented out) ---
# fpr, tpr, thresholds = roc_curve(y, oof_proba)
# roc_auc = auc(fpr, tpr)
# plt.figure(figsize=(6, 5))
# plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random classifier')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve — Logistic Regression (OOF)')
# plt.legend()
# plt.tight_layout()
# plt.show()

# --- Threshold optimisation for accuracy ---
threshold_grid = np.linspace(0, 1, 1000)
accuracies     = np.array([((oof_proba >= t) == y.values).mean() for t in threshold_grid])
best_threshold = threshold_grid[np.argmax(accuracies)]

# print(f"Default threshold (0.50) accuracy : {((oof_proba >= 0.5) == y.values).mean():.4f}")
# print(f"Optimal threshold ({best_threshold:.4f}) accuracy: {accuracies.max():.4f}")


### Optuna Tuning — LGBMClassifier on Churn

#### Hyperparameter search space rationale

| Parameter | Range | Scale | Reasoning |
|---|---|---|---|
| `n_estimators` | 100–1000 | linear | Covers everything from a quick shallow ensemble to a deep one. Too few → underfitting; too many → diminishing returns (though LightGBM regularises well). |
| `learning_rate` | 0.01–0.3 | **log** | Log scale because the difference between 0.01 and 0.05 is far more meaningful than between 0.25 and 0.30. Lower rates need more trees, which the wide `n_estimators` range accommodates. |
| `num_leaves` | 20–300 | linear | The primary complexity knob in LightGBM (leaf-wise growth). Default is 31; going up to 300 lets Optuna explore much more expressive trees without imposing an artificial ceiling. |
| `max_depth` | 3–12 | linear | Informed by the max_depth plot above — accuracy plateaus or degrades beyond a certain depth, so searching past 12 is wasteful. |
| `min_child_samples` | 10–200 | linear | Minimum samples required in a leaf. Higher values regularise against overfitting on small data segments; the wide range lets Optuna trade off bias vs variance freely. |
| `subsample` | 0.5–1.0 | linear | Row bagging fraction. Values below 0.5 make each tree too different from the data; 1.0 means no bagging. |
| `colsample_bytree` | 0.5–1.0 | linear | Feature bagging fraction per tree — analogous to `max_features` in Random Forests. Same lower bound reasoning as `subsample`. |
| `reg_alpha` | 1e-8–10 | **log** | L1 regularisation. Log scale lets us probe near-zero (effectively off) and genuinely large values with equal density. |
| `reg_lambda` | 1e-8–10 | **log** | L2 regularisation — same reasoning as `reg_alpha`. |
| `min_split_gain` | 0–1 | linear | Minimum loss reduction required to make a split. Zero means any split is allowed; higher values prune away weak splits and act as additional regularisation. |

In [11]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def churn_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,  # must be >0 for subsample to take effect
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
        'verbose':           -1,
    }

    cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        LGBMClassifier(**params), X, y,
        cv=cv_inner, scoring='accuracy',
    )
    return scores.mean()


churn_study = optuna.create_study(direction='maximize')
churn_study.optimize(churn_objective, n_trials=50, show_progress_bar=True)

print(f"Best CV accuracy : {churn_study.best_value:.4f}")
print(f"Best params:\n{churn_study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy : 0.8617
Best params:
{'n_estimators': 781, 'learning_rate': 0.0742530521244504, 'num_leaves': 272, 'max_depth': 5, 'min_child_samples': 27, 'subsample': 0.872040275761442, 'colsample_bytree': 0.6186626716830201, 'reg_alpha': 0.4906904516573272, 'reg_lambda': 0.008363221879018322, 'min_split_gain': 0.6056247723173562}


### Optuna Tuning — XGBClassifier

| Parameter | Range | Scale | Reasoning |
|---|---|---|---|
| `n_estimators` | 100–1000 | linear | Same role as in LightGBM — number of boosting rounds. |
| `learning_rate` | 0.01–0.3 | **log** | Step-size shrinkage; log scale for the same reason as LGBM. |
| `max_depth` | 3–10 | linear | XGBoost grows depth-wise (unlike LGBM's leaf-wise), so depths above 10 are rarely beneficial and slow to train. |
| `min_child_weight` | 1–20 | linear | Minimum sum of instance weights in a child — XGBoost's analogue of `min_child_samples`. |
| `subsample` | 0.5–1.0 | linear | Row subsampling fraction per tree. |
| `colsample_bytree` | 0.5–1.0 | linear | Column subsampling fraction per tree. |
| `reg_alpha` | 1e-8–10 | **log** | L1 regularisation on leaf weights. |
| `reg_lambda` | 1e-8–10 | **log** | L2 regularisation on leaf weights. |
| `gamma` | 0–1 | linear | Minimum loss reduction required to make a split — XGBoost's equivalent of `min_split_gain`. |

In [12]:
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 1.0),
        'verbosity':        0,
        'eval_metric':      'logloss',
    }
    cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        XGBClassifier(**params), X, y,
        cv=cv_inner, scoring='accuracy',
    )
    return scores.mean()


xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f'Best CV accuracy : {xgb_study.best_value:.4f}')
print(f'Best params : {xgb_study.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy : 0.8618
Best params : {'n_estimators': 858, 'learning_rate': 0.09049341790650128, 'max_depth': 4, 'min_child_weight': 11, 'subsample': 0.9387008587548251, 'colsample_bytree': 0.6367683094020606, 'reg_alpha': 3.4374679108614363, 'reg_lambda': 0.021454953674135722, 'gamma': 0.29590332314192086}


### Optuna Tuning — CatBoostClassifier

CatBoost uses symmetric (oblivious) trees and a Bayesian bootstrap by default,
so its parameter names and semantics differ slightly from LGBM and XGBoost.

| Parameter | Range | Scale | Reasoning |
|---|---|---|---|
| `iterations` | 100–1000 | linear | Number of trees — equivalent to `n_estimators`. |
| `learning_rate` | 0.01–0.3 | **log** | Same step-size role as in the other boosters. |
| `depth` | 3–10 | linear | CatBoost caps symmetric tree depth at 16; practical gains plateau well before that. |
| `l2_leaf_reg` | 1–10 | **log** | L2 regularisation on leaf values — CatBoost's primary regulariser. |
| `min_data_in_leaf` | 1–100 | linear | Minimum samples per leaf — prevents tiny, overfit leaves. |
| `random_strength` | 0.1–10 | **log** | Amount of randomness added when scoring splits; higher values add more exploration and act as regularisation. |
| `bagging_temperature` | 0–1 | linear | Controls the Bayesian bootstrap: 0 = deterministic (no bagging), 1 = standard Bayesian bootstrap. |

In [13]:
from catboost import CatBoostClassifier


def catboost_objective(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 100, 1000),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':               trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'min_data_in_leaf':    trial.suggest_int('min_data_in_leaf', 1, 100),
        'random_strength':     trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'verbose':             0,
    }
    cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        CatBoostClassifier(**params), X, y,
        cv=cv_inner, scoring='accuracy',
    )
    return scores.mean()


catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50, show_progress_bar=True)

print(f'Best CV accuracy : {catboost_study.best_value:.4f}')
print(f'Best params : {catboost_study.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]

Best CV accuracy : 0.8619
Best params : {'iterations': 978, 'learning_rate': 0.2974043596459918, 'depth': 3, 'l2_leaf_reg': 4.503108741810998, 'min_data_in_leaf': 60, 'random_strength': 0.34127125268775854, 'bagging_temperature': 0.018519692170337304}


### Feature Importance Analysis

#### What are SHAP values?

SHAP (SHapley Additive exPlanations) roots feature importance in cooperative game theory. The idea: treat each feature as a *player* in a coalition game where the *payout* is the model's prediction for a specific row. The Shapley value for a feature is its **average marginal contribution** to the prediction, averaged across every possible ordering in which features could be introduced.

**What a SHAP value tells you for a given row and feature:**
- **Positive** → this feature pushed the prediction *above* the dataset average (toward Churn = 1).
- **Negative** → this feature pushed the prediction *below* the dataset average (toward Churn = 0).
- The SHAP values for all features in a row sum exactly to `prediction − baseline`, where the baseline is the average prediction over the training set.

**Why SHAP over the other importance types?**

| Measure | Scope | Limitation |
|---|---|---|
| **Split count** | Global | Biased toward high-cardinality features — more splits ≠ more useful |
| **Gain** | Global | Better than split count, but still one number per feature averaged across all trees |
| **Permutation** | Global | Model-agnostic and honest, but unreliable when features are correlated |
| **SHAP** | **Per-sample** | Exact (for trees via TreeSHAP), consistent, and works at both the individual prediction and global level |

The **beeswarm summary plot** shows one dot per sample per feature. The x-axis is the SHAP value; colour encodes the raw feature value (red = high, blue = low). Features are ranked by mean |SHAP| so the most globally impactful features appear at the top.

In [14]:
best_params = dict(churn_study.best_params)
best_params.update({'subsample_freq': 1, 'verbose': -1})

# --- model fit and importance plots (commented out) ---
# X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# best_model = LGBMClassifier(**best_params)
# best_model.fit(X_tr, y_tr)
# feature_names = np.array(encoded_features)
# gain_imp  = best_model.booster_.feature_importance(importance_type='gain')
# split_imp = best_model.booster_.feature_importance(importance_type='split')
# fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(feature_names) * 0.3)))
# for ax, imp, title in zip(
#     axes,
#     [gain_imp, split_imp],
#     ['Gain Importance', 'Split Count Importance'],
# ):
#     order = np.argsort(imp)
#     ax.barh(feature_names[order], imp[order])
#     ax.set_title(title)
#     ax.set_xlabel('Importance')
# plt.tight_layout()
# plt.show()


In [15]:
# import shap

# X_shap = X_val.sample(2000, random_state=42)

# explainer = shap.TreeExplainer(best_model)
# shap_vals = explainer.shap_values(X_shap)

# if isinstance(shap_vals, list):
#     shap_vals = shap_vals[1]

# shap.summary_plot(shap_vals, X_shap, feature_names=encoded_features)

In [16]:
# mean_abs_shap = np.abs(shap_vals).mean(axis=0)
# order = np.argsort(mean_abs_shap)

# fig, ax = plt.subplots(figsize=(8, max(6, len(feature_names) * 0.3)))
# ax.barh(feature_names[order], mean_abs_shap[order])
# ax.set_title('Mean |SHAP| Value')
# ax.set_xlabel('Mean |SHAP|')
# plt.tight_layout()
# plt.show()

In [17]:
# perm = permutation_importance(
#     best_model, X_val, y_val,
#     n_repeats=10, scoring='accuracy', random_state=42,
# )

# order = np.argsort(perm.importances_mean)
# fig, ax = plt.subplots(figsize=(8, max(6, len(feature_names) * 0.3)))
# ax.barh(
#     feature_names[order],
#     perm.importances_mean[order],
#     xerr=perm.importances_std[order],
# )
# ax.set_title('Permutation Importance (held-out 20%, 10 repeats)')
# ax.set_xlabel('Mean accuracy decrease ± std')
# plt.tight_layout()
# plt.show()

## Save Artifacts

Persist the processed dataframes, best LGBM parameters, the fitted logistic
regression pipeline, and the optimal classification threshold to disk so they
can be loaded by downstream modelling notebooks without re-running this notebook.


In [18]:
import pyarrow as pa
import pyarrow.parquet as pq

os.makedirs('./data/processed', exist_ok=True)
os.makedirs('./data/models',    exist_ok=True)

# --- processed dataframes (written via pyarrow directly to avoid a pandas/pyarrow
#     version mismatch in pandas' patch_pyarrow() wrapper) ---
pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False),
               './data/processed/train_df.parquet')
pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False),
               './data/processed/test_df.parquet')
print('Saved: data/processed/train_df.parquet')
print('Saved: data/processed/test_df.parquet')

# --- best LGBM hyperparameters (Churn study) ---
lgbm_params_out = dict(churn_study.best_params)
lgbm_params_out['subsample_freq'] = 1
with open('./data/models/lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_params_out, f, indent=2)
print('Saved: data/models/lgbm_best_params.json')

# --- LR pipeline fitted on the full training set ---
lr_pipeline_full = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42)),
])
lr_pipeline_full.fit(X, y)
joblib.dump(lr_pipeline_full, './data/models/lr_pipeline.pkl')
print('Saved: data/models/lr_pipeline.pkl')

# --- optimal classification threshold ---
with open('./data/models/lr_best_threshold.json', 'w') as f:
    json.dump({'best_threshold': float(best_threshold)}, f, indent=2)
print('Saved: data/models/lr_best_threshold.json')
# --- best XGBoost hyperparameters ---
with open('./data/models/xgb_best_params.json', 'w') as f:
    json.dump(dict(xgb_study.best_params), f, indent=2)
print('Saved: data/models/xgb_best_params.json')

# --- best CatBoost hyperparameters ---
with open('./data/models/catboost_best_params.json', 'w') as f:
    json.dump(dict(catboost_study.best_params), f, indent=2)
print('Saved: data/models/catboost_best_params.json')


Saved: data/processed/train_df.parquet
Saved: data/processed/test_df.parquet
Saved: data/models/lgbm_best_params.json
Saved: data/models/lr_pipeline.pkl
Saved: data/models/lr_best_threshold.json
Saved: data/models/xgb_best_params.json
Saved: data/models/catboost_best_params.json


## OOF Prediction Probabilities

Generate out-of-fold positive-class probabilities for each tuned model using
5-fold `StratifiedKFold`. Saving probabilities (rather than hard labels) keeps
downstream options open: probability calibration, threshold sweeps, and
ensemble stacking all need the raw scores.

> **Runtime note**: each `cross_val_predict` call trains 5 models on ~475k rows.
> CatBoost is the slowest of the three; expect several minutes per model.

In [19]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- LGBM (best_params already includes subsample_freq=1 and verbose=-1) ---
lgbm_oof = cross_val_predict(
    LGBMClassifier(**best_params), X, y,
    cv=cv5, method='predict_proba',
)[:, 1]

# --- XGBoost ---
xgb_full_params = dict(xgb_study.best_params)
xgb_full_params.update({'verbosity': 0, 'eval_metric': 'logloss'})
xgb_oof = cross_val_predict(
    XGBClassifier(**xgb_full_params), X, y,
    cv=cv5, method='predict_proba',
)[:, 1]

# --- CatBoost ---
catboost_full_params = dict(catboost_study.best_params)
catboost_full_params['verbose'] = 0
catboost_oof = cross_val_predict(
    CatBoostClassifier(**catboost_full_params), X, y,
    cv=cv5, method='predict_proba',
)[:, 1]

np.save('./data/processed/lgbm_oof_proba.npy',     lgbm_oof)
np.save('./data/processed/xgb_oof_proba.npy',      xgb_oof)
np.save('./data/processed/catboost_oof_proba.npy', catboost_oof)

for name, arr in [('LGBM', lgbm_oof), ('XGB', xgb_oof), ('CatBoost', catboost_oof)]:
    acc = (arr >= 0.5) == y.values
    print(f'{name:10s}  mean proba: {arr.mean():.4f}  OOF accuracy: {acc.mean():.4f}')

LGBM        mean proba: 0.2252  OOF accuracy: 0.8617
XGB         mean proba: 0.2252  OOF accuracy: 0.8620
CatBoost    mean proba: 0.2252  OOF accuracy: 0.8617


## Overfitting Checks

Three checks to verify the Optuna-tuned `best_model` is not overfitting.

### Check 1 — Train vs. validation accuracy gap

In [20]:
# train_acc = accuracy_score(y_tr, best_model.predict(X_tr))
# val_acc   = accuracy_score(y_val, best_model.predict(X_val))

# print(f'Train accuracy : {train_acc:.4f}')
# print(f'Val   accuracy : {val_acc:.4f}')
# print(f'Gap            : {train_acc - val_acc:.4f}')

### Check 2 — 5-fold OOF with best params

The Optuna objective used 3-fold CV, so `churn_study.best_value` may be
slightly optimistic. A fresh 5-fold run on the full dataset gives a more
stable and honest estimate.

In [21]:
# cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scores_5fold = cross_val_score(
#     LGBMClassifier(**best_params), X, y,
#     cv=cv5, scoring='accuracy',
# )

# print(f'Optuna 3-fold CV score : {churn_study.best_value:.4f}')
# print(f'5-fold OOF CV score    : {scores_5fold.mean():.4f} ± {scores_5fold.std():.4f}')

### Check 3 — Learning curve

Plots train and CV accuracy as training-set size grows. A persistently large
gap across all sizes indicates structural overfitting; converging curves mean
the model is data-limited rather than over-complex.

3-fold CV and 8 evenly-spaced size checkpoints keep runtime manageable on
the full 594k-row dataset.

In [22]:
# from sklearn.model_selection import learning_curve

# train_sizes, train_scores, val_scores = learning_curve(
#     LGBMClassifier(**best_params),
#     X, y,
#     cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
#     train_sizes=np.linspace(0.1, 1.0, 8),
#     scoring='accuracy',
#     n_jobs=-1,
# )

# train_mean = train_scores.mean(axis=1)
# val_mean   = val_scores.mean(axis=1)
# train_std  = train_scores.std(axis=1)
# val_std    = val_scores.std(axis=1)

# plt.figure(figsize=(7, 4))
# plt.plot(train_sizes, train_mean, marker='o', label='Train')
# plt.plot(train_sizes, val_mean,   marker='o', label='CV (3-fold)')
# plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
# plt.fill_between(train_sizes, val_mean   - val_std,   val_mean   + val_std,   alpha=0.15)
# plt.xlabel('Training set size')
# plt.ylabel('Accuracy')
# plt.title('Learning Curve — Best LGBMClassifier')
# plt.legend()
# plt.tight_layout()
# plt.show()